# IPW Effect Estimation and Post-Weighting Balance

## Scenario
Following the propensity modeling step, you will apply **inverse probability weighting (IPW)**
to estimate the causal effect of the promotional offer on user upgrade behavior.
You will also check whether IPW successfully reduces covariate imbalance,
and compare the IPW estimate to the naive (unadjusted) estimate.

## Your task
You will produce:
1. IPW weights computed from the propensity scores
2. SMD table comparing balance **before and after** IPW
3. IPW Average Treatment Effect (ATE) for **both** `upgraded_to_new_tier` and `new_tier_monthly_revenue`
4. Naive ATE (simple mean difference) for comparison
5. A written justification for which estimate is more credible and why

## Requirements
- Use `observational_offer_adoption.csv`
- Refit the propensity model with the same covariates as the previous exercise
- IPW weights: treated = 1/propensity, control = 1/(1 − propensity)
- Report both naive ATE and IPW ATE numerically
- The written comparison must reference the SMD improvement

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from pathlib import Path
PROJECT_DIR = Path(r"/content/drive/Othercomputers/My PC/Desktop/All_Work_Related/Udacity/Applied_Statistics_ND/Project")
DATA_DIR = PROJECT_DIR / "Project_Data"
if not DATA_DIR.exists():
    DATA_DIR = Path("Project_Data")

CHOICE_PATH  = DATA_DIR / "choice_conjoint_tasks.csv"
BUNDLES_PATH = DATA_DIR / "candidate_bundles.csv"
ADOPT_PATH   = DATA_DIR / "observational_offer_adoption.csv"
SURVEY_PATH  = DATA_DIR / "survey_value_drivers.csv"
MMM_PATH     = DATA_DIR / "marketing_weekly_channels.csv"
print("DATA_DIR:", DATA_DIR.resolve())

DATA_DIR: /content/drive/Othercomputers/My PC/Desktop/All_Work_Related/Udacity/Applied_Statistics_ND/Project/Project_Data


In [6]:
# ── Step 1: Load data and refit propensity model ──────────────────────────
df = pd.read_csv(ADOPT_PATH)

# Encode covariates (same setup as propensity diagnostics exercise)
df_enc = pd.get_dummies(df, columns=['region','device'], drop_first=True)
encoded_cat_cols = [c for c in df_enc.columns if c.startswith(('region_','device_'))]
numeric_covs = ['age','tenure_days','prior_spend_12m','sessions_30d']
all_covs = numeric_covs + encoded_cat_cols

X = df_enc[all_covs].astype(float)
y = df_enc['offer_received']

# Refit propensity model
prop_model = LogisticRegression(max_iter=500, random_state=42)
prop_model.fit(X, y)
df_enc['propensity'] = prop_model.predict_proba(X)[:, 1]
print("Propensity score range:", round(df_enc['propensity'].min(), 4), "–", round(df_enc['propensity'].max(), 4))

Propensity score range: 0.2628 – 0.8821


In [7]:
# ── Step 2: Compute IPW weights ───────────────────────────────────────────
# IPW weights rebalance the sample by up-weighting under-represented observations
df_enc['ipw'] = np.where(
    df_enc['offer_received'] == 1,
    1.0 / df_enc['propensity'],
    1.0 / (1.0 - df_enc['propensity'])
)
print("IPW weight summary:")
print(df_enc['ipw'].describe().round(3))

IPW weight summary:
count    50000.000
mean         2.000
std          0.394
min          1.203
25%          1.702
50%          1.929
75%          2.205
max          8.478
Name: ipw, dtype: float64


In [8]:
# ── Step 3: Post-IPW balance check (SMD) ──────────────────────────────────
def weighted_mean(series, weights):
    return np.average(series, weights=weights)

def weighted_smd(col, df_full, treatment_col):
    t = df_full[df_full[treatment_col] == 1]
    c = df_full[df_full[treatment_col] == 0]
    mt = weighted_mean(t[col], t['ipw'])
    mc = weighted_mean(c[col], c['ipw'])
    pooled_std = np.sqrt((t[col].var() + c[col].var()) / 2 + 1e-9)
    return (mt - mc) / pooled_std

# Unweighted SMD (pre-IPW)
treated = df_enc[df_enc['offer_received'] == 1]
control = df_enc[df_enc['offer_received'] == 0]
def smd_raw(col, t, c):
    pooled_std = np.sqrt((t[col].var() + c[col].var()) / 2 + 1e-9)
    return (t[col].mean() - c[col].mean()) / pooled_std

smd_before = pd.Series({v: smd_raw(v, treated, control) for v in numeric_covs})
smd_after  = pd.Series({v: weighted_smd(v, df_enc, 'offer_received') for v in numeric_covs})

balance = pd.DataFrame({'SMD Before': smd_before, 'SMD After (IPW)': smd_after})
print("Covariate Balance (SMD):")
print(balance.round(4))

Covariate Balance (SMD):
                 SMD Before  SMD After (IPW)
age                  0.0067           0.0001
tenure_days          0.2824          -0.0003
prior_spend_12m      0.0687          -0.0057
sessions_30d         0.0697           0.0005


In [9]:
# ── Step 4: IPW ATE and naive ATE — for BOTH outcomes ─────────────────────
# The project requires ATEs for both upgrade rate and revenue.
# Always report both: a significant upgrade rate but tiny revenue effect
# tells a very different story than a significant effect on both.

def naive_ate(outcome, treated_df, control_df):
    return treated_df[outcome].mean() - control_df[outcome].mean()

def ipw_ate(outcome, treated_df, control_df):
    return (weighted_mean(treated_df[outcome], treated_df['ipw']) -
            weighted_mean(control_df[outcome], control_df['ipw']))

treated = df_enc[df_enc['offer_received'] == 1]
control = df_enc[df_enc['offer_received'] == 0]

for outcome, label, unit in [
    ('upgraded_to_new_tier',    'Upgrade rate',    'pp'),
    ('new_tier_monthly_revenue','Monthly revenue', '$/user'),
]:
    n_ate = naive_ate(outcome, treated, control)
    i_ate = ipw_ate(outcome, treated, control)
    scale = 100 if unit == 'pp' else 1
    fmt   = '.2f' if unit == 'pp' else '.4f'
    print(f"{label}:")
    print(f"  Naive ATE: {n_ate*scale:{fmt}} {unit}")
    print(f"  IPW ATE:   {i_ate*scale:{fmt}} {unit}")
    print(f"  Difference (naive − IPW): {(n_ate - i_ate)*scale:{fmt}} → selection bias")
    print()


Upgrade rate:
  Naive ATE: 3.68 pp
  IPW ATE:   2.02 pp
  Difference (naive − IPW): 1.66 → selection bias

Monthly revenue:
  Naive ATE: 0.4154 $/user
  IPW ATE:   0.2381 $/user
  Difference (naive − IPW): 0.1773 → selection bias



## Credibility Assessment

In the markdown cell below, answer:
1. Did the SMD improve after IPW? Reference specific before/after values.
2. How much does the naive ATE differ from the IPW ATE? What does this tell you about
   the direction and magnitude of selection bias in this dataset?
3. Which estimate would you present to a stakeholder, and what one-sentence caveat
   would you include about the remaining assumptions?

In [10]:
# Summarize both ATEs and credibility
print("Balance summary:")
print(f"  Max |SMD| before IPW: {balance['SMD Before'].abs().max():.4f}")
print(f"  Max |SMD| after IPW:  {balance['SMD After (IPW)'].abs().max():.4f}")
print()
print("Effect estimates (IPW-adjusted):")
print(f"  Upgrade rate ATE:   +{0.0202*100:.2f} pp  (naive: +{0.0368*100:.2f} pp)")
print(f"  Revenue ATE:        +${0.2381:.4f}/user  (naive: +${0.4154:.4f}/user)")
print()
print("Interpretation:")
print(f"  Naive estimates overstate effects by ~{(0.0368-0.0202)*100:.1f} pp (upgrade)")
print(f"  and ~${(0.4154-0.2381):.2f}/user (revenue) due to selection bias.")
print("  Remaining assumption: no unmeasured confounders (unconfoundedness).")
print("  Treat IPW ATEs as directional estimates, not precise causal claims.")


Balance summary:
  Max |SMD| before IPW: 0.2824
  Max |SMD| after IPW:  0.0057

Effect estimates (IPW-adjusted):
  Upgrade rate ATE:   +2.02 pp  (naive: +3.68 pp)
  Revenue ATE:        +$0.2381/user  (naive: +$0.4154/user)

Interpretation:
  Naive estimates overstate effects by ~1.7 pp (upgrade)
  and ~$0.18/user (revenue) due to selection bias.
  Remaining assumption: no unmeasured confounders (unconfoundedness).
  Treat IPW ATEs as directional estimates, not precise causal claims.
